<a href="https://colab.research.google.com/github/xaviergs/ratpenats/blob/main/Ratpenats_al_Cap_de_Creus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inicialització de l'entorn

In [1]:
# Instal·lem el client de Supabase
!pip install supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 1.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from supabase import create_client, Client

# 1. Recuperem credencials dels Secrets de Colab
try:
    url = userdata.get('SUPABASE_URL')
    key = userdata.get('SUPABASE_KEY')

    # 2. Inicialitzem el client
    supabase: Client = create_client(url, key)
    print("✅ Client de Supabase inicialitzat correctament.")

    # 3. Verificació real: Intentem llegir la taula file_registry
    # Això ens confirmarà que la taula existeix i els permisos (RLS) són correctes
    test_query = supabase.table("file_registry").select("count", count='exact').execute()
    print(f"✅ Connexió verificada. La taula 'file_registry' té actualment {test_query.count} registres.")

except Exception as e:
    print(f"❌ Error en la verificació: {e}")

✅ Client de Supabase inicialitzat correctament.
✅ Connexió verificada. La taula 'file_registry' té actualment 347 registres.


In [3]:
from google.colab import drive
import os

# 1. Muntem el Drive (et sortirà un avís per donar permís)
drive.mount('/content/drive')

# 2. Defineix l'ID de la carpeta mare del projecte
# Recorda: l'ID és el codi que surt a la URL de la carpeta al navegador
PARENT_FOLDER_ID = userdata.get('PARENT_FOLDER_ID')

print("✅ Drive muntat i carpeta mare configurada.")

Mounted at /content/drive
✅ Drive muntat i carpeta mare configurada.


In [4]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build

# Creem el servei de l'API de Google Drive
# Això permet al programa fer cerques per ID i llistar fitxers recursivament
try:
    drive_service = build('drive', 'v3')
    print("✅ Variable 'drive_service' creada correctament.")
except Exception as e:
    print(f"❌ Error creant el servei de Drive: {e}")

✅ Variable 'drive_service' creada correctament.


# Funcions de càrrega

In [5]:
# --- CONFIGURACIÓ DE LOCALITZACIONS ---
# Baixem les localitzacions actives de la base de dades per mapejar els directoris de Drive
try:
    res_loc = supabase.table("locations").select("id, location_code").eq("is_active", True).execute()
    # Creem un diccionari: {'nom_carpeta': id_base_dades}
    location_map = {item['location_code']: item['id'] for item in res_loc.data}
    print(f"✅ Sincronitzades {len(location_map)} localitzacions actives de Supabase.")
except Exception as e:
    print(f"❌ Error sincronitzant localitzacions: {e}")
    location_map = {}

✅ Sincronitzades 23 localitzacions actives de Supabase.


In [6]:
def get_all_files_in_subfolders(parent_id):
    # 0. Obtenim el nom de la carpeta mare de forma segura
    try:
        res = drive_service.files().get(fileId=parent_id, fields='name').execute()
        parent_folder_name = res['name'] if isinstance(res, dict) else res
    except:
        parent_folder_name = "Folder_Principal" # Nom per defecte si falla

    # 1. Busquem les carpetes de les localitzacions (LOCATIONS) al Drive mare
    query_folders = f"'{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    folders_result = drive_service.files().list(q=query_folders, fields="files(id, name)").execute()
    subfolders = folders_result.get('files', [])

    all_files_data = []

    for folder in subfolders:
        location_name = folder['name']

        # Filtre: si la carpeta no està al mapa, l'ignorem
        if location_name not in location_map:
            continue

        print(f"🔍 Explorant localització vàlida: {location_name}...")

        # 2. Busquem els continguts dins de la carpeta de la localització
        query_contents = f"'{folder['id']}' in parents and trashed = false"
        contents_result = drive_service.files().list(q=query_contents, fields="files(id, name, mimeType, modifiedTime)").execute()
        items = contents_result.get('files', [])

        for item in items:
            # CAS A: Fitxer CSV directe
            if item['mimeType'] != 'application/vnd.google-apps.spreadsheet' and item['name'].lower().endswith('.csv'):
                all_files_data.append({
                    'drive_id': item['id'],
                    'file_name': item['name'],
                    'location': location_name,
                    'parent_folder_name': parent_folder_name, # <-- AFEGIT
                    'last_modified_drive': item['modifiedTime']
                })

            # CAS B: Subcarpeta (ex: per dates)
            elif item['mimeType'] == 'application/vnd.google-apps.folder':
                query_inner = f"'{item['id']}' in parents and trashed = false"
                inner_result = drive_service.files().list(q=query_inner, fields="files(id, name, mimeType, modifiedTime)").execute()
                inner_items = inner_result.get('files', [])

                for inner_item in inner_items:
                    if inner_item['mimeType'] != 'application/vnd.google-apps.spreadsheet' and inner_item['name'].lower().endswith('.csv'):
                        all_files_data.append({
                            'drive_id': inner_item['id'],
                            'file_name': inner_item['name'],
                            'location': location_name,
                            'parent_folder_name': parent_folder_name, # <-- AFEGIT
                            'last_modified_drive': inner_item['modifiedTime']
                        })

    return all_files_data

In [7]:
# --- REGISTRE DE NOUS FITXERS ---
def register_new_files(new_files):
    if not new_files:
        print("✨ No hi ha fitxers nous per registrar.")
        return

    records = []
    for f in new_files:
        # Obtenim l'ID numèric de la localització usant el mapa que hem creat abans
        loc_id = location_map.get(f['location'])

        records.append({
            "drive_id": f['drive_id'],
            "file_name": f['file_name'],
            "location_id": loc_id, # ARA GUARDEM L'ID RELACIONAL
            "status": "pending",
            "last_modified_drive": f['last_modified_drive']
        })

    try:
        supabase.table("file_registry").insert(records).execute()
        print(f"✅ S'han registrat {len(records)} fitxers nous a 'file_registry'.")
    except Exception as e:
        print(f"❌ Error registrant fitxers: {e}")

# Executem la comparació
#files_to_process = get_new_files(files_found)

#print(f"📊 Balanç de fitxers:")
#print(f"- Total trobats a les carpetes de LOCATIONS: {len(files_found)}")
#print(f"- Fitxers que ja estaven registrats: {len(files_found) - len(files_to_process)}")
#print(f"- FITXERS NOUS PER PROCESSAR: {len(files_to_process)}")

#if files_to_process:
#    print("\nPropers fitxers a entrar a la base de dades:")
#    for f in files_to_process[:5]: # Mostrem els 5 primers per comprovar
#        print(f"  • {f['file_name']} (Localització: {f['location']})")
#    if len(files_to_process) > 5:
#        print(f"  ... i {len(files_to_process) - 5} més.")

In [8]:
def get_new_files(drive_files):
    """
    Compara la llista de Drive amb la base de dades i retorna només els fitxers
    que no s'han registrat mai (o que no estan completats).
    """
    try:
        # 1. Consultem els IDs que ja existeixen a Supabase
        # Només ens cal la columna 'drive_id' per comparar
        response = supabase.table("file_registry").select("drive_id").execute()

        # Creem un conjunt (set) d'IDs existents per a una cerca ultra ràpida
        existing_ids = {record['drive_id'] for record in response.data}

        # 2. Filtrem: Només ens quedem amb els fitxers de Drive que NO estan a la BD
        new_files = [f for f in drive_files if f['drive_id'] not in existing_ids]

        print(f"📊 Balanç: {len(drive_files)} fitxers al Drive, {len(existing_ids)} ja a la BD. Nous per processar: {len(new_files)}")

        return new_files

    except Exception as e:
        print(f"⚠️ Error consultant fitxers existents: {e}")
        # En cas d'error, per seguretat, retornem la llista buida per no duplicar dades
        return []

In [9]:
import pandas as pd
from io import BytesIO
from googleapiclient.http import MediaIoBaseDownload

def read_sheet_to_dataframe(file_id):
    # Per a CSVs (fitxers binaris), fem servir get_media en comptes d'export_media
    request = drive_service.files().get_media(fileId=file_id)

    fh = BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while done is False:
        status, done = downloader.next_chunk()

    fh.seek(0)
    # Intentem llegir el CSV. Si els teus CSVs usen separador ';' en comptes de coma,
    # pots afegir sep=';' aquí dins.
    return pd.read_csv(fh)

# Funcions de recompte

In [10]:
def count_species_and_buzz_per_datetime(df_mostres, column_name='MANUAL ID', date_column='DATE', hour_column='HOUR', location_name='', buzz_keyword='BUZZ', no_noise=True):
    """
    Processa el CSV combinant DATE i TIME per obtenir timestamps reals.
    Retorna un DataFrame compatible amb la taula 'bat_observations' de Supabase.
    """
    try:
        # 1. Selecció de columnes necessàries
        # Incloem 'TIME' si existeix per tenir precisió de minuts/segons
        needed_cols = [column_name, date_column, hour_column, 'TIME']
        cols_present = [c for c in needed_cols if c in df_mostres.columns]
        df_sub = df_mostres[cols_present].copy()

        # 2. Creació del Timestamp real (temp_dt)
        if date_column in df_sub.columns and 'TIME' in df_sub.columns:
            # Combinem DATE i TIME en una sola cadena i convertim
            df_sub['temp_dt'] = pd.to_datetime(
                df_sub[date_column].astype(str) + ' ' + df_sub['TIME'].astype(str),
                errors='coerce'
            )
        elif date_column in df_sub.columns and hour_column in df_sub.columns:
            # Si no hi ha TIME, sumem l'hora a la data
            df_sub['temp_dt'] = pd.to_datetime(df_sub[date_column], errors='coerce') + \
                                pd.to_timedelta(pd.to_numeric(df_sub[hour_column], errors='coerce').fillna(0), unit='h')
        else:
            # Cas base: només data
            df_sub['temp_dt'] = pd.to_datetime(df_sub[date_column], errors='coerce')

        # Netegem files on la data hagi fallat
        df_sub = df_sub.dropna(subset=['temp_dt'])

        # 3. Mapeig de la Localització (ID numèric)
        loc_id = location_map.get(location_name)
        if loc_id is None:
            print(f"⚠️ Localització '{location_name}' no trobada al mapa d'IDs.")
            return pd.DataFrame()

        records = []

        # 4. Processament de files (Lògica d'espècies i BUZZ)
        for _, row in df_sub.iterrows():
            if pd.isna(row[column_name]): continue

            # Netegem i separem els ítems per comes (Ex: "Pipistrellus, BUZZ")
            items = [item.strip() for item in str(row[column_name]).split(',')]
            last_species = None
            species_counts = {}
            species_buzz_counts = {}

            for item in items:
                if no_noise and item.lower() == 'noise': continue

                if item == buzz_keyword:
                    if last_species:
                        species_buzz_counts[last_species] = species_buzz_counts.get(last_species, 0) + 1
                else:
                    species_counts[item] = species_counts.get(item, 0) + 1
                    if item not in species_buzz_counts: species_buzz_counts[item] = 0
                    last_species = item

            # Guardem els registres amb l'hora ja extreta correctament
            for species in species_counts.keys():
                records.append({
                    'observation_timestamp': row['temp_dt'].strftime('%Y-%m-%d %H:%M:%S'),
                    'observation_date': row['temp_dt'].strftime('%Y-%m-%d'),
                    'observation_hour': int(row['temp_dt'].hour), # Ara l'hora serà la real del CSV
                    'location_id': loc_id,
                    'species': species,
                    'count': species_counts[species],
                    'buzz': species_buzz_counts.get(species, 0)
                })

        if not records: return pd.DataFrame()

        # 5. Agrupació final per evitar duplicats
        df_final = pd.DataFrame(records)
        df_final = df_final.groupby(
            ['observation_timestamp', 'observation_date', 'observation_hour', 'location_id', 'species'],
            as_index=False
        ).sum()

        return df_final

    except Exception as e:
        print(f"❌ Error en la transformació: {e}")
        import traceback
        traceback.print_exc()
        return None

In [11]:
def process_all_files(files_to_process, debug_limit=None, dry_run=True):
    """
    Gestiona el flux complet: llegir de Drive, transformar i pujar a Supabase.
    """
    list_to_run = files_to_process[:debug_limit] if debug_limit else files_to_process

    if not list_to_run:
        print("☕ No hi ha fitxers nous per processar.")
        return

    processed_dataframes = []

    print(f"🚀 Iniciant {'SIMULACIÓ' if dry_run else 'PROCESSAMENT REAL'} de {len(list_to_run)} fitxers...")

    for f in list_to_run:
        try:
            print(f"\n📄 Fitxer: {f['file_name']} (Localització: {f['location']})")

            # 1. Llegir (Fem servir la funció que ja tenies)
            df_raw = read_sheet_to_dataframe(f['drive_id'])

            # 2. Transformar (ATENCIÓ: ara usem el paràmetre 'location_name')
            df_clean = count_species_and_buzz_per_datetime(
                df_raw,
                location_name=f['location']
            )

            if df_clean is not None and not df_clean.empty:
                print(f"✅ Transformació ok! ({len(df_clean)} registres d'observació generats)")
                display(df_clean.head(5))

                processed_dataframes.append(df_clean)

                if not dry_run:
                    # 3. Pujar a la taula d'observacions
                    data_to_insert = df_clean.to_dict(orient='records')
                    supabase.table("bat_observations").insert(data_to_insert).execute()

                    # 4. Actualitzar el registre (ATENCIÓ: ara usem .update() no .insert())
                    # Com que el fitxer ja s'ha registrat com a 'pending' abans, ara l'actualitzem a 'completed'
                    supabase.table("file_registry").update({
                        "status": "completed",
                        "error_details": None
                    }).eq("drive_id", f['drive_id']).execute()

                    print("💾 Dades guardades i registre actualitzat.")
                else:
                    print("🧪 Mode simulació: No s'ha guardat res.")
            else:
                print("⚠️ El fitxer no ha generat dades o no s'ha trobat la localització.")

        except Exception as e:
            print(f"❌ ERROR processant {f['file_name']}: {e}")
            if not dry_run:
                # Si falla, ho deixem escrit a la base de dades perquè sàpigues per què
                supabase.table("file_registry").update({
                    "status": "error",
                    "error_details": str(e)
                }).eq("drive_id", f['drive_id']).execute()

    return processed_dataframes

# --- EXECUCIÓ DE PROVA ---
# Ara pots executar-ho així per veure què passa sense por a trencar res:
#resultats = process_all_files(files_to_process, debug_limit=2, dry_run=True)

In [12]:
def update_file_status(file_info, status="completed", error_msg=None):
    """
    Registra o actualitza l'estat d'un fitxer individual a file_registry.
    """
    record = {
        "drive_id": file_info['drive_id'],
        "file_name": file_info['file_name'],
        "location_id": location_map.get(file_info['location']),
        "status": status,
        "error_details": error_msg,
        "processed_at": "now()" # O deixa que Supabase posi el DEFAULT NOW()
    }

    try:
        # Fem servir upsert per registrar-lo per primer cop amb el seu estat final
        supabase.table("file_registry").upsert(record, on_conflict="drive_id").execute()
    except Exception as e:
        print(f"❌ Error fatal actualitzant file_registry: {e}")

In [13]:
def insert_observations_to_supabase(df_clean, drive_id, folder_name):
    """
    Insereix les dades ja transformades a la taula 'bat_observations'.
    df_clean ja ha de venir amb els noms de columnes correctes de la funció de transformació.
    """
    try:
        # 1. Obtenir el ID de la localització des del mapeig (per seguretat)
        loc_id = location_map.get(folder_name)

        if not loc_id:
            return False, f"Localització '{folder_name}' no trobada al mapa."

        # 2. Preparar les dades
        # Afegim la referència al fitxer d'origen a cada fila
        data_to_insert = df_clean.to_dict(orient='records')
        for record in data_to_insert:
            record['file_drive_id'] = drive_id
            # El location_id i els noms de data ja venen posats de la funció de transformació

        # 3. Executar inserció massiva
        if data_to_insert:
            supabase.table("bat_observations").insert(data_to_insert).execute()

        # 4. Actualitzar l'estat del fitxer al registre
        supabase.table("file_registry").update({
            "status": "completed",
            "location_id": loc_id,
            "error_details": None
        }).eq("drive_id", drive_id).execute()

        return True, None

    except Exception as e:
        # En cas d'error, el registrem a la taula file_registry
        supabase.table("file_registry").update({
            "status": "error",
            "error_details": str(e)
        }).eq("drive_id", drive_id).execute()
        return False, str(e)

## Processament de Dades i Inserció Relacional
Aquest bloc realitza la iteració sobre els fitxers identificats al Google Drive.
Per a cada fitxer, es realitza una neteja de dades, es valida la seva pertinença a una
localització activa i es vincula amb l'ID corresponent de la base de dades de Supabase,
garantint la integritat referencial per a futures anàlisis amb dades meteorològiques.

In [14]:
# --- EXECUCIÓ PRINCIPAL ---

# 1. Busquem tots els fitxers al Drive
all_files = get_all_files_in_subfolders(PARENT_FOLDER_ID)

# 2. Filtrem els que realment són nous comparant amb Supabase
files_to_process = get_new_files(all_files)


🔍 Explorant localització vàlida: LaValleta_Llança...
🔍 Explorant localització vàlida: Aiguamolls_Corral_Guatlles...
🔍 Explorant localització vàlida: Aiguamolls_Closes_Sant_Joan...
🔍 Explorant localització vàlida: Aiguamolls_Estany_Sant_Joan...
🔍 Explorant localització vàlida: Aiguamolls_Lliris_Grocs...
🔍 Explorant localització vàlida: CdC_Norfeu...
🔍 Explorant localització vàlida: CdC_MasMares...
🔍 Explorant localització vàlida: CdC_Canadell...
🔍 Explorant localització vàlida: Muga_Asil...
🔍 Explorant localització vàlida: Muga_Trabuc...
🔍 Explorant localització vàlida: Muga_Resclosa...
🔍 Explorant localització vàlida: Muga_Vil_Cast...
🔍 Explorant localització vàlida: Aiguamolls_Tec...
🔍 Explorant localització vàlida: CdC_Comes_Tortes...
🔍 Explorant localització vàlida: CdC_Mas_Margall...
🔍 Explorant localització vàlida: Aiguamolls_EstanysPau...
🔍 Explorant localització vàlida: CdC_Montjoi...
🔍 Explorant localització vàlida: CdC_Olivar...
🔍 Explorant localització vàlida: CdC_Ponac...
🔍 

In [15]:
# 2. Verifiquem si hi ha feina a fer
if not files_to_process:
    print("✨ No hi ha fitxers nous per processar. Tot està al dia!")
else:
    total_files = len(files_to_process)
    print(f"🚀 S'han trobat {total_files} fitxers nous. Començant el procés...")

    # Utilitzem enumerate per tenir l'índex 'i' (comença a 0)
    for i, f in enumerate(files_to_process, start=1):
        # Missatge amb el comptador: [1 / 10] Processant...
        print(f"📦 [{i} / {total_files}] Processant: {f['parent_folder_name']} / {f['location']} / {f['file_name']}")

        try:
            # --- PAS NOU: Pre-registre obligatori per la Clau Forana ---
            update_file_status(f, status="processing")

            # A. Procés de dades
            df_raw = read_sheet_to_dataframe(f['drive_id'])
            df_clean = count_species_and_buzz_per_datetime(df_raw, location_name=f['location'])

            if df_clean is not None and not df_clean.empty:
                # B. Inserció a observations
                success, err = insert_observations_to_supabase(df_clean, f['drive_id'], f['location'])

                if success:
                    update_file_status(f, status="completed")
                    print("   ✅ Fet.")
                else:
                    update_file_status(f, status="error", error_msg=f"Error BD: {err}")
                    print(f"   ⚠️ Error en inserció: {err}")
            else:
                update_file_status(f, status="completed", error_msg="Sense observacions")
                print("   ℹ️ Saltat (sense dades).")

        except Exception as e:
            update_file_status(f, status="error", error_msg=str(e))
            print(f"   ❌ Error crític: {e}")

    print("\n🏁 Procés finalitzat.")

🚀 S'han trobat 16 fitxers nous. Començant el procés...
📦 [1 / 16] Processant: Ratpenats / Aiguamolls_Corral_Guatlles / L005202_2026_03_07-91a7fe19997b194df57744b1dd106813.csv
   ✅ Fet.
📦 [2 / 16] Processant: Ratpenats / Aiguamolls_Closes_Sant_Joan / L004850_2026_03_12-87d754c7821e02c8cac9e04a65e2e5a2.csv
   ✅ Fet.
📦 [3 / 16] Processant: Ratpenats / Aiguamolls_Estany_Sant_Joan / L003009_2026_03_12-f012e7f494ec1854e950fd838c037e95.csv
   ✅ Fet.
📦 [4 / 16] Processant: Ratpenats / Aiguamolls_Estany_Sant_Joan / ID_L003009_2026_01_08.csv
   ℹ️ Saltat (sense dades).
📦 [5 / 16] Processant: Ratpenats / Aiguamolls_Lliris_Grocs / L002197_2026_03_20-89faea8f69fee0529cdddc0e81818781.csv
   ✅ Fet.
📦 [6 / 16] Processant: Ratpenats / CdC_Norfeu / L004191_2026_03_10-879b308adb7264d4fc01f0d0685598d7.csv
   ✅ Fet.
📦 [7 / 16] Processant: Ratpenats / CdC_MasMares / L003629_2026_03_10-0e820a8a4e9b538f47db3cda37113062.csv
   ✅ Fet.
📦 [8 / 16] Processant: Ratpenats / Muga_Vil_Cast / L002190_2026_03_12-b8ea387

# Extracció de dades meteorològiques de la XEMA

In [16]:
def get_weather_metrics_map():
    """
    Llegeix weather_metrics de Supabase i crea un mapa:
    { target_column: {'id': id, 'agg': aggregation} }
    Exemple: { 'temp_c': {'id': 32, 'agg': 'mean'}, 'precip_mm': {'id': 35, 'agg': 'sum'} }
    """
    try:
        # Afegim 'aggregation' a la selecció (i mantenim 'id' com a codi XEMA)
        res = supabase.table("weather_metrics").select("id, target_column, aggregation").execute()

        if not res.data:
            print("⚠️ Alerta: La taula weather_metrics està buida.")
            return {}

        # Creem el diccionari amb la nova estructura
        mapping = {
            row['target_column']: {
                'id': row['id'],
                'agg': row.get('aggregation', 'mean') # Si és buit, per defecte 'mean'
            }
            for row in res.data
            if row.get('target_column')
        }

        print(f"📊 Mètriques configurades a Supabase: {list(mapping.keys())}")
        return mapping

    except Exception as e:
        print(f"❌ Error carregant el mapatge dinàmic: {e}")
        return {}

In [17]:
w_map = get_weather_metrics_map()
w_map

📊 Mètriques configurades a Supabase: ['temp_c', 'rel_humidity', 'precip_mm', 'wind_speed_2m', 'wind_speed_10m']


{'temp_c': {'id': 32, 'agg': 'mean'},
 'rel_humidity': {'id': 33, 'agg': 'mean'},
 'precip_mm': {'id': 35, 'agg': 'sum'},
 'wind_speed_2m': {'id': 46, 'agg': 'mean'},
 'wind_speed_10m': {'id': 30, 'agg': 'mean'}}

In [18]:
def fetch_xema_data(station_code, target_date, metrics_map):
    """
    Descarrega dades de la XEMA.
    Atura el procés si es detecta error de quota (429).
    """
    base_url = "https://api.meteo.cat/xema/v1/variables/mesurades"
    headers = {"x-api-key": userdata.get('XEMA_API_KEY'), "Accept": "application/json"}

    day_data = {h: {
        "station_code": station_code,
        "observation_date": target_date.isoformat(),
        "observation_hour": h
    } for h in range(24)}

    print(f"📡 Consultant estació {station_code} ({target_date})...")

    for col_name, config in metrics_map.items():
        xema_id = config['id']
        agg_type = config['agg']

        y, m, d = target_date.year, f"{target_date.month:02d}", f"{target_date.day:02d}"
        url = f"{base_url}/{xema_id}/{y}/{m}/{d}"
        params = {"codiEstacio": station_code}

        try:
            response = requests.get(url, headers=headers, params=params, timeout=20)

            # --- CONTROL DE QUOTA (429) ---
            if response.status_code == 429:
                print("\n🛑 ERROR 429: S'ha superat el límit de quota de l'API Meteocat.")
                # Lancem una excepció per aturar tota l'execució del script
                raise PermissionError("Límit de quota excedit. Aturant el procés global.")

            # GESTIÓ SILENCIOSA per variables no disponibles
            if response.status_code in [400, 404]:
                continue

            response.raise_for_status()
            lectures = response.json().get("lectures", [])

            if not lectures:
                continue

            df = pd.DataFrame(lectures)
            df["data"] = pd.to_datetime(df["data"])

            df_hourly = df.set_index("data").resample("1h").agg({"valor": agg_type}).reset_index()

            for _, row in df_hourly.iterrows():
                h = row["data"].hour
                if h in day_data and pd.notnull(row["valor"]):
                    day_data[h][col_name] = round(float(row["valor"]), 2)

            time.sleep(0.4)

        except PermissionError:
            # Re-lancem l'error de quota perquè el main_sync_process també s'aturi
            raise
        except Exception as e:
            print(f"  ❌ Error inesperat en mètrica {col_name} (ID {xema_id}): {e}")

    final_rows = [vals for vals in day_data.values() if len(vals) > 3]
    print(f"✅ Processades {len(final_rows)} hores per a {station_code}.")
    return final_rows

In [19]:
def save_weather_observations(observations_list):
    """
    Si el registre existeix (mateixa estació/data/hora), l'actualitza.
    Si no existeix, el crea.
    """
    if not observations_list:
        return None

    try:
        # L'argument 'on_conflict' és clau si tens diverses claus úniques,
        # però amb la configuració actual, Supabase ho detecta sol.
        res = supabase.table("weather_observations").upsert(
            observations_list,
            on_conflict="station_code, observation_date, observation_hour"
        ).execute()

        return res.data

    except Exception as e:
        print(f"❌ Error en l'operació upsert: {e}")
        return None

In [20]:
def get_station_metrics(station_code):
    """
    Consulta les metadades d'una estació per saber quines variables mesura
    i quins IDs utilitza.
    """
    url = f"https://api.meteo.cat/xema/v1/estacions/{station_code}/metadades"
    headers = {
        "x-api-key": userdata.get('XEMA_API_KEY'),
        "Accept": "application/json"
    }

    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()

        variables = r.json().get("variables", [])

        print(f"📊 Metadades per a l'estació: {station_code}")
        print(f"{'ID':<5} | {'Nom de la Variable':<40} | {'Unitat'}")
        print("-" * 60)

        metrics_found = {}
        for v in variables:
            print(f"{v['codi']:<5} | {v['nom']:<40} | {v['unitat']}")
            metrics_found[v['nom']] = v['codi']

        return metrics_found, variables

    except Exception as e:
        print(f"❌ Error obtenint metadades per {station_code}: {e}")
        return None

In [21]:
from datetime import date, datetime
import time # També el necessitem per al time.sleep()

def main_sync_process(max_days=25):
    """
    Orquestra la càrrega de dades: consulta dies pendents des de la View,
    prioritzant els dies més recents, i processa un màxim de 'max_days'.
    S'atura immediatament si es detecta falta de quota.
    """
    print(f"🚀 Iniciant procés de sincronització inversa (Màxim {max_days} dies)...")

    # 1. Carreguem configuració de mètriques
    metrics_map = get_weather_metrics_map()
    if not metrics_map:
        print("❌ No s'ha pogut carregar la configuració de mètriques. Avortant.")
        return

    # 2. Consultem quins dies i estacions falten
    try:
        res = supabase.table("missing_weather_days")\
            .select("*")\
            .order("observation_date", desc=True)\
            .execute()
        pending_work = res.data
    except Exception as e:
        print(f"❌ Error consultant la View de dies pendents: {e}")
        return

    if not pending_work:
        print("✅ Tot està al dia! No hi ha dades pendents de carregar.")
        return

    print(f"📅 Total de tasques pendents trobades: {len(pending_work)}")

    # 3. Bucle de processament
    unique_dates = sorted(list(set(item['observation_date'] for item in pending_work)), reverse=True)
    dates_to_process = unique_dates[:max_days]

    days_count = 0
    for current_date_str in dates_to_process:
        target_date = date.fromisoformat(current_date_str)
        stations_for_day = [item['station_code'] for item in pending_work if item['observation_date'] == current_date_str]

        print(f"\n--- 📅 Processant dia {days_count + 1}/{len(dates_to_process)}: {target_date} ---")

        for st_code in stations_for_day:
            try:
                # A. Descarregar
                data_to_save = fetch_xema_data(st_code, target_date, metrics_map)

                # B. Gravar a Supabase
                if data_to_save:
                    save_weather_observations(data_to_save)
                    print(f"  ✅ Estació {st_code}: Gravades {len(data_to_save)} hores.")
                else:
                    print(f"  ⚠️ Estació {st_code}: No s'han obtingut dades.")

            except PermissionError as e:
                # --- AQUÍ ÉS ON ATUREM EL WRAPPER ---
                print(f"\n🛑 CRITICAL: Aturant tot el procés per falta de quota (429).")
                print(f"Última acció intentada: {st_code} el dia {target_date}")
                return # Surt completament de la funció main_sync_process

            except Exception as e:
                print(f"  ❌ Error processant estació {st_code}: {e}")
                continue

        days_count += 1
        time.sleep(1)

    print(f"\n✨ Procés finalitzat.")
    if dates_to_process:
        print(f"📊 S'ha treballat el rang: des de {dates_to_process[0]} fins a {dates_to_process[-1]}")
    print("Si encara queden dades, torna a executar aquesta cel·la.")

In [22]:
# --- EXECUCIÓ ---
import pandas as pd
import requests
import time
from datetime import date, datetime
from google.colab import userdata
main_sync_process(max_days=100)

🚀 Iniciant procés de sincronització inversa (Màxim 100 dies)...
📊 Mètriques configurades a Supabase: ['temp_c', 'rel_humidity', 'precip_mm', 'wind_speed_2m', 'wind_speed_10m']
📅 Total de tasques pendents trobades: 9

--- 📅 Processant dia 1/8: 2026-03-20 ---
📡 Consultant estació W1 (2026-03-20)...
✅ Processades 24 hores per a W1.
  ✅ Estació W1: Gravades 24 hores.

--- 📅 Processant dia 2/8: 2026-03-13 ---
📡 Consultant estació W1 (2026-03-13)...
✅ Processades 24 hores per a W1.
  ✅ Estació W1: Gravades 24 hores.

--- 📅 Processant dia 3/8: 2026-03-12 ---
📡 Consultant estació W1 (2026-03-12)...
✅ Processades 24 hores per a W1.
  ✅ Estació W1: Gravades 24 hores.

--- 📅 Processant dia 4/8: 2026-03-11 ---
📡 Consultant estació D4 (2026-03-11)...
✅ Processades 24 hores per a D4.
  ✅ Estació D4: Gravades 24 hores.

--- 📅 Processant dia 5/8: 2026-03-10 ---
📡 Consultant estació D4 (2026-03-10)...
✅ Processades 24 hores per a D4.
  ✅ Estació D4: Gravades 24 hores.
📡 Consultant estació VZ (2026-03-1

# Gravació de resultats agrupats

In [23]:
import gspread
import pandas as pd
from google.auth import default
from gspread_dataframe import set_with_dataframe
from googleapiclient.discovery import build

def export_view_to_sheets():
    """
    Llegeix la vista 'bat_observations_full' de Supabase i actualitza
    la pestanya 'Dades agrupades' del Google Sheet.
    """
    print("🚀 Iniciant exportació a la pestanya 'Dades agrupades'...")

    # 1. Autenticació i Connexió
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    drive_service = build('drive', 'v3', credentials=creds)

    # 2. Lectura de Supabase amb paginació
    all_data = []
    chunk_size = 1000
    start = 0

    print("📥 Baixant dades de Supabase...")
    while True:
        try:
            response = supabase.table('bat_observations_full') \
                .select('*') \
                .range(start, start + chunk_size - 1) \
                .execute()

            chunk = response.data
            if not chunk:
                break

            all_data.extend(chunk)

            if len(chunk) < chunk_size:
                break
            start += chunk_size
        except Exception as e:
            print(f"❌ Error a Supabase: {e}")
            return

    if not all_data:
        print("⚠️ No hi ha dades per exportar.")
        return

    df = pd.DataFrame(all_data)
    print(f"📊 Registres recuperats: {len(df)}")

    # 3. Gestió del fitxer a Google Drive
    file_name = "Comptatge històric combinat"
    folder_id = "1mhbDzg_3xAbK7N8yf9X24Q-0Ggpn_-av"

    try:
        sh = gc.open(file_name)
    except gspread.exceptions.SpreadsheetNotFound:
        print(f"🆕 Creant nou fitxer...")
        sh = gc.create(file_name)
        file_id = sh.id
        file = drive_service.files().get(fileId=file_id, fields='parents').execute()
        previous_parents = ",".join(file.get('parents'))
        drive_service.files().update(
            fileId=file_id, addParents=folder_id, removeParents=previous_parents
        ).execute()

    # 4. Gestió específica de la pestanya "Dades agrupades"
    try:
        # Intentem obrir la pestanya pel seu nom
        worksheet = sh.worksheet("Dades agrupades")
        print(f"📖 Pestanya 'Dades agrupades' trobada.")
    except gspread.exceptions.WorksheetNotFound:
        # Si no existeix, la creem
        print(f"➕ Creant nova pestanya 'Dades agrupades'...")
        worksheet = sh.add_worksheet(title="Dades agrupades", rows="100", cols="20")

    # 5. Volcat de dades
    try:
        worksheet.clear()
        set_with_dataframe(worksheet, df, include_index=False)
        print(f"✅ Sincronització completada: {len(df)} línies a 'Dades agrupades'.")
    except Exception as e:
        print(f"❌ Error en escriure al Sheet: {e}")

In [24]:
def export_weather_history_to_sheets():
    """
    Recupera les dades de la vista 'weather_observations_history'
    i les exporta a la pestanya 'Històric Meteorologia'.
    """
    print("🚀 Iniciant exportació de l'històric temporal...")

    # 1. Autenticació (Aprofitem les credencials ja definides al script)
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # 2. Lectura de dades de Supabase amb paginació
    # Com que és un històric, és probable que hi hagi moltes dades.
    print("📥 Baixant dades de la vista 'weather_observations_history'...")
    all_history = []
    chunk_size = 1000
    start = 0

    while True:
        try:
            # Fem la query a la vista que m'has indicat
            response = supabase.table('weather_observations_history') \
                .select('*') \
                .range(start, start + chunk_size - 1) \
                .execute()

            chunk = response.data
            if not chunk:
                break

            all_history.extend(chunk)

            if len(chunk) < chunk_size:
                break
            start += chunk_size
        except Exception as e:
            print(f"❌ Error consultant l'històric a Supabase: {e}")
            return

    if not all_history:
        print("⚠️ No s'ha trobat cap dada a la vista 'weather_observations_history'.")
        return

    df_history = pd.DataFrame(all_history)
    print(f"📊 Registres recuperats: {len(df_history)}")

    # 3. Gestió del Google Sheet
    file_name = "Comptatge històric combinat"
    try:
        sh = gc.open(file_name)

        # Intentem obrir o crear la pestanya específica per a l'històric
        try:
            worksheet = sh.worksheet("Històric Meteorologia")
            print("📖 Pestanya 'Històric Meteorologia' trobada.")
        except gspread.exceptions.WorksheetNotFound:
            print("➕ Creant nova pestanya 'Històric Meteorologia'...")
            worksheet = sh.add_worksheet(title="Històric Meteorologia", rows="100", cols="20")

        # 4. Volcat de les dades
        worksheet.clear()
        set_with_dataframe(worksheet, df_history, include_index=False)
        print(f"✅ Sincronització completada amb èxit a 'Històric Meteorologia'.")

    except Exception as e:
        print(f"❌ Error en la gestió del Google Sheet: {e}")



In [25]:
# Per executar l'exportació manualment:
export_view_to_sheets()

🚀 Iniciant exportació a la pestanya 'Dades agrupades'...
📥 Baixant dades de Supabase...
📊 Registres recuperats: 6887
📖 Pestanya 'Dades agrupades' trobada.
✅ Sincronització completada: 6887 línies a 'Dades agrupades'.


In [26]:
export_weather_history_to_sheets()

🚀 Iniciant exportació de l'històric temporal...
📥 Baixant dades de la vista 'weather_observations_history'...
📊 Registres recuperats: 7608
📖 Pestanya 'Històric Meteorologia' trobada.
✅ Sincronització completada amb èxit a 'Històric Meteorologia'.


# Funcions auxiliars

In [27]:

## AUX per obtenir totes les estacions de la XEMA
url = f"https://api.meteo.cat/xema/v1/estacions/metadades?estat=ope&data=2026-01-01Z"
headers = {
    "x-api-key": userdata.get('XEMA_API_KEY'),
    "Accept": "application/json"
}
r = requests.get(url, headers=headers, timeout=15)
rjson = r.json()

In [28]:
## AUX per obtenir el consum de dades
url = f"https://api.meteo.cat/quotes/v1/consum-actual"
headers = {
    "x-api-key": userdata.get('XEMA_API_KEY'),
    "Accept": "application/json"
}
r = requests.get(url, headers=headers, timeout=15)
rjson = r.json()

In [29]:
rjson

{'client': {'nom': '848 Xavier Garcia Sabater'},
 'plans': [{'nom': 'Predicció_100',
   'periode': 'Mensual',
   'maxConsultes': 100,
   'consultesRestants': 100,
   'consultesRealitzades': 0},
  {'nom': 'Referència Bàsic',
   'periode': 'Mensual',
   'maxConsultes': 2000,
   'consultesRestants': 2000,
   'consultesRealitzades': 0},
  {'nom': 'XDDE_250',
   'periode': 'Mensual',
   'maxConsultes': 250,
   'consultesRestants': 250,
   'consultesRealitzades': 0},
  {'nom': 'XEMA_750 OD',
   'periode': 'Mensual',
   'maxConsultes': 750,
   'consultesRestants': 750,
   'consultesRealitzades': 0},
  {'nom': 'Quota',
   'periode': 'Mensual',
   'maxConsultes': 300,
   'consultesRestants': 300,
   'consultesRealitzades': 0}]}

In [30]:
## AUX per saber les metadades que recull una estació
codiEstacio='D4'
url = f"https://api.meteo.cat/xema/v1/estacions/{codiEstacio}/variables/mesurades/metadades?estat=ope&amp;data=2026-01-01Z"
headers = {
    "x-api-key": userdata.get('XEMA_API_KEY'),
    "Accept": "application/json"
}
r = requests.get(url, headers=headers, timeout=15)
rjson = r.json()

In [31]:
from datetime import date

def test_full_station_sync(station_code, target_date):
    print(f"🔍 Iniciant test definitiu per a l'estació: {station_code} el dia {target_date}")

    # 1. Carreguem el mapa de mètriques (per saber què hem de demanar a la XEMA)
    print("--- Pas 1: Carregant configuració de mètriques ---")
    metrics_map = get_weather_metrics_map()
    if not metrics_map:
        print("❌ Error: No s'ha pogut carregar el mapa de mètriques.")
        return
    print(f"✅ Mètriques trobades: {list(metrics_map.keys())}")

    # 2. Descarreguem les dades de l'API de la XEMA
    print(f"--- Pas 2: Descarregant dades de la XEMA ---")
    raw_data = fetch_xema_data(station_code, target_date, metrics_map)

    if not raw_data:
        print(f"⚠️ No s'han trobat dades per a {station_code} en aquesta data.")
        return
    print(f"✅ S'han obtingut {len(raw_data)} registres horaris.")

    # 3. Guardem/Actualitzem a Supabase
    print(f"--- Pas 3: Injectant a la taula 'weather_observations' ---")
    result = save_weather_observations(raw_data)

    if result is not None:
        print(f"🚀 ÈXIT TOTAL! S'han processat {len(result)} files correctament.")
        # Mostrem la primera fila del resultat com a prova
        print(f"Exemple de dada guardada: {result[0]}")
    else:
        print("❌ El procés de guardat ha fallat.")

# --- EXECUCIÓ DEL TEST ---
# Escull l'estació i el dia que vulguis (ex: Roses 'UN' o Cabanes 'UG')
TEST_STATION = "D4"
TEST_DATE = date(2025, 6, 1)

test_full_station_sync(TEST_STATION, TEST_DATE)

🔍 Iniciant test definitiu per a l'estació: D4 el dia 2025-06-01
--- Pas 1: Carregant configuració de mètriques ---
📊 Mètriques configurades a Supabase: ['temp_c', 'rel_humidity', 'precip_mm', 'wind_speed_2m', 'wind_speed_10m']
✅ Mètriques trobades: ['temp_c', 'rel_humidity', 'precip_mm', 'wind_speed_2m', 'wind_speed_10m']
--- Pas 2: Descarregant dades de la XEMA ---
📡 Consultant estació D4 (2025-06-01)...
✅ Processades 24 hores per a D4.
✅ S'han obtingut 24 registres horaris.
--- Pas 3: Injectant a la taula 'weather_observations' ---
🚀 ÈXIT TOTAL! S'han processat 24 files correctament.
Exemple de dada guardada: {'id': 10177, 'station_code': 'D4', 'observation_date': '2025-06-01', 'observation_hour': 0, 'temp_c': 18.35, 'rel_humidity': 82.0, 'precip_mm': 0.0, 'wind_speed_2m': None, 'wind_speed_10m': 0.3}


# Visualització i anàlisi

In [32]:
def get_bat_observations_dataset(include_null_meteo=True, max_total_records=10000):
    """
    Carrega dades de la View fent paginació per saltar el límit de 1000 de Supabase.
    """
    all_data = []
    page_size = 1000
    current_offset = 0

    print(f"📥 Iniciant descàrrega paginada (màxim {max_total_records} registres)...")

    try:
        while current_offset < max_total_records:
            # Calculem el final del rang per a aquesta pàgina
            range_to = current_offset + page_size - 1

            # Construïm la query per a aquest bloc
            query = supabase.table("bat_observations_full").select("*")

            if not include_null_meteo:
                query = query.not_.is_("temp", "null")

            # Demanem el rang específic
            res = query.range(current_offset, range_to).execute()

            batch_data = res.data
            if not batch_data: # Si no tornen més dades, hem arribat al final real
                break

            all_data.extend(batch_data)
            print(f"   ✅ Bloc rebut: {len(all_data)} files carregades...")

            # Si el bloc és més petit que el page_size, és que no hi ha més dades
            if len(batch_data) < page_size:
                break

            current_offset += page_size

        df = pd.DataFrame(all_data)

        if df.empty:
            return df

        # Neteja i formats (Data + Hora)
        df['observation_date'] = pd.to_datetime(df['observation_date'])
        df['timestamp'] = pd.to_datetime(
            df['observation_date'].dt.strftime('%Y-%m-%d') + ' ' +
            df['observation_hour'].astype(str).str.zfill(2) + ':00:00'
        )

        print(f"\n🚀 Procés finalitzat: {len(df)} registres totals a la memòria.")
        return df

    except Exception as e:
        print(f"❌ Error durant la paginació: {e}")
        return pd.DataFrame()

In [33]:
def prepare_dataset_chronology(df, start_hour=18):
    """
    Afegeix una columna d'ordenació al dataset perquè la nit
    comenci a l'hora desitjada (per defecte les 18h).
    """
    # Creem el mapeig: Hora -> Ordre
    # Si start_hour és 18, l'ordre serà: 18:0, 19:1, ..., 23:5, 0:6, 1:7...
    hours_order = [(h % 24) for h in range(start_hour, start_hour + 24)]
    sort_mapping = {hour: index for index, hour in enumerate(hours_order)}

    # Assignem l'índex d'ordenació
    df['hour_sort_index'] = df['observation_hour'].map(sort_mapping)

    # També podem crear una etiqueta de text que forci l'ordre en gràfics
    # Ex: "18h", "19h"... perquè Plotly no les ordeni numèricament
    df['hour_label'] = df['observation_hour'].apply(lambda x: f"{x:02d}h")

    # Ordenem el dataframe físicament per aquesta nova columna
    df = df.sort_values('hour_sort_index').reset_index(drop=True)

    print(f"✅ Cronologia preparada. La nit comença a les {start_hour}:00h.")

    # Afegeix això després de carregar el dataset
    df['year_month'] = df['observation_date'].dt.strftime('%Y-%m')

    return df

In [34]:
# Carreguem només dades que ja tinguin meteo per poder correlacionar
df = get_bat_observations_dataset(include_null_meteo=False)
df = prepare_dataset_chronology(df, start_hour=14)

# 1. Veure les primeres files
display(df.head())

# 2. Resum estadístic (mitjanes de temp, max de counts, etc.)
display(df.describe())

# 3. Quantes observacions tenim per espècie?
print(df['species'].value_counts())

📥 Iniciant descàrrega paginada (màxim 10000 registres)...
   ✅ Bloc rebut: 1000 files carregades...
   ✅ Bloc rebut: 2000 files carregades...
   ✅ Bloc rebut: 3000 files carregades...
   ✅ Bloc rebut: 4000 files carregades...
   ✅ Bloc rebut: 5000 files carregades...
   ✅ Bloc rebut: 6000 files carregades...
   ✅ Bloc rebut: 6887 files carregades...

🚀 Procés finalitzat: 6887 registres totals a la memòria.
✅ Cronologia preparada. La nit comença a les 14:00h.


,observation_date,observation_hour,species,total_count,total_buzz,iad,location_name,station_name,temp,rel_humidity,precip_mm,wind_speed,timestamp,hour_sort_index,hour_label,year_month
0,2025-12-10,16,PpygMin,2,0,0.000000,Estanys de Vilaüt/Pau,Castelló d'Empúries,13.30,87.0,0.0,NaN,2025-12-10 16:00:00,2,16h,2025-12
1,2024-11-18,16,Pkuhnat,50,3,6.000000,Ponac,Roses,14.60,66.0,0.0,0.95,2024-11-18 16:00:00,2,16h,2024-11
2,2025-11-12,16,PpygMin,19,1,5.263158,Ponac,Roses,17.95,99.5,0.0,4.35,2025-11-12 16:00:00,2,16h,2025-11
3,2026-01-08,16,Pippip,1,0,0.000000,Closes de Sant Joan,Castelló d'Empúries,10.35,73.0,0.0,NaN,2026-01-08 16:00:00,2,16h,2026-01
4,2025-12-16,16,TadNyc,2,0,0.000000,Montjoi,Roses,12.55,100.0,0.0,0.95,2025-12-16 16:00:00,2,16h,2025-12


,observation_date,observation_hour,total_count,total_buzz,iad,temp,rel_humidity,precip_mm,wind_speed,timestamp,hour_sort_index
count,6887,6887.000000,6887.000000,6887.000000,6887.000000,6887.000000,6887.00000,6887.000000,2169.000000,6887,6887.000000
mean,2025-02-24 17:57:51.409902848,13.763903,35.101496,3.193263,2.327515,17.309837,84.65885,0.010077,1.388128,2025-02-25 07:43:41.460723456,8.144911
min,2023-06-05 00:00:00,0.000000,1.000000,0.000000,0.000000,2.150000,12.50000,0.000000,0.100000,2023-06-05 19:00:00,2.000000
25%,2024-09-05 00:00:00,3.000000,2.000000,0.000000,0.000000,13.350000,74.00000,0.000000,0.650000,2024-09-05 23:00:00,5.000000
50%,2025-05-14 00:00:00,19.000000,4.000000,0.000000,0.000000,17.550000,90.00000,0.000000,0.900000,2025-05-14 20:00:00,8.000000
75%,2025-08-26 12:00:00,21.000000,16.000000,0.000000,0.000000,21.300000,100.00000,0.000000,1.400000,2025-08-26 23:30:00,11.000000
max,2026-03-20 00:00:00,23.000000,708.000000,849.000000,300.000000,29.650000,100.00000,5.000000,6.350000,2026-03-20 22:00:00,16.000000
std,NaN,8.788476,99.597128,27.383497,10.678469,5.334513,16.82996,0.160950,1.291507,NaN,3.326012


species
PpygMin      1873
Pippip       1192
Pkuhnat      1165
TadNyc        783
EptNycVes     710
Myo50         367
Hypsav        236
Rhifer        176
PleSp         163
Rhihip         84
Barbar         54
NoID           44
Myo30          30
Mnat            5
Rhieur          3
Pipkuh          1
BarSp           1
Name: count, dtype: int64


In [35]:
import plotly.express as px
import plotly.graph_objects as go

import plotly.graph_objects as go

def plot_nightly_cycle_combined(df):
    """
    Pinta l'activitat (count) i la caça (buzz) en una nit contínua.
    Requereix que el df hagi passat per prepare_dataset_chronology().
    """
    # Agreguem dades mantenint l'ordre cronològic
    # Agrupem per l'índex d'ordre i l'etiqueta per no perdre la seqüència
    hourly = df.groupby(['hour_sort_index', 'hour_label'])[['count', 'buzz']].mean().reset_index()

    fig = go.Figure()

    # 1. Barres per al Comptatge Total (Activitat)
    fig.add_trace(go.Bar(
        x=hourly['hour_label'],
        y=hourly['count'],
        name='Activitat (Deteccions)',
        marker_color='rgba(100, 149, 237, 0.6)', # Blau suau transparent
        offsetgroup=1
    ))

    # 2. Línia per als Buzzes (Comportament de caça)
    # Posem els buzzes en un segon eix Y (a la dreta) per veure millor la tendència
    fig.add_trace(go.Scatter(
        x=hourly['hour_label'],
        y=hourly['buzz'],
        name='Caça (Feeding Buzz)',
        line=dict(color='orange', width=3, shape='spline'), # 'spline' per suavitzar la línia
        yaxis='y2'
    ))

    # Configuració de l'estil i els eixos
    fig.update_layout(
        title='<b>Cicle Nocturn: Activitat General vs. Intensitat de Caça</b>',
        xaxis_title='Hora de la nit (ordre cronològic)',
        yaxis_title='Mitjana de Deteccions (Counts)',
        yaxis2=dict(
            title='Mitjana de Feeding Buzz',
            overlaying='y',
            side='right',
            showgrid=False
        ),
        template='plotly_dark',
        hovermode='x unified', # Permet veure els dos valors alhora en passar el ratolí
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.show()

plot_nightly_cycle_combined(df)


KeyError: "Columns not found: 'count', 'buzz'"

In [ ]:
def plot_nocturnal_cycle_filtered(df, species=None, location=None):
    """
    Genera la gràfica de cicle nocturn permetent filtrar per espècie o lloc.
    """
    dff = df.copy()

    # Apliquem filtres si s'han especificat
    title_suffix = ""
    if species:
        dff = dff[dff['species'] == species]
        title_suffix += f" - Espècie: {species}"
    if location:
        dff = dff[dff['sampling_spot'] == location]
        title_suffix += f" - Lloc: {location}"

    if dff.empty:
        print("⚠️ No hi ha dades per a aquest filtre.")
        return

    # Agreguem (com ja tenim l'ordre al dataset, és directe)
    hourly = dff.groupby(['hour_sort_index', 'hour_label'])[['count', 'buzz']].mean().reset_index()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=hourly['hour_label'], y=hourly['count'], name='Activitat', marker_color='rgba(100, 149, 237, 0.6)'))
    fig.add_trace(go.Scatter(x=hourly['hour_label'], y=hourly['buzz'], name='Buzz', line=dict(color='orange', width=3), yaxis='y2'))

    fig.update_layout(
        title=f'<b>Cicle Nocturn Filtrat</b><br><span style="font-size:12px;">{title_suffix}</span>',
        xaxis_title='Hora de la nit',
        yaxis_title='Mitjana de Counts',
        yaxis2=dict(title='Mitjana de Buzz', overlaying='y', side='right', showgrid=False),
        template='plotly_dark'
    )
    fig.show()

In [ ]:
import plotly.graph_objects as go

def plot_observations_vs_temperature_interactive(df):
    """
    Genera un gràfic de dispersió interactiu que relaciona l'activitat (deteccions)
    amb la temperatura, permetent filtrar per espècie mitjançant botons natius.
    """
    # 1. Obtenir la llista completa d'espècies presents al dataset
    species_list = sorted(df['species'].unique().tolist())

    fig = go.Figure()

    # 2. Afegir una traça per a cada espècie
    # Les mantenim totes creades, però podem controlar la seva visibilitat
    for sp in species_list:
        dff_sp = df[df['species'] == sp]
        fig.add_trace(
            go.Scatter(
                x=dff_sp['temp'],
                y=dff_sp['count'],
                mode='markers',
                name=sp,
                visible=True, # Per defecte es mostren totes en carregar
                text=dff_sp.apply(lambda r: f"Lloc: {r['sampling_spot']}<br>Data: {r['observation_date'].date()}", axis=1),
                marker=dict(
                    size=8,
                    opacity=0.5,
                    line=dict(width=0.5, color='white')
                )
            )
        )

    # 3. Configuració dels botons de filtratge d'espècies
    # El botó "Totes" activa la visibilitat de totes les traces
    species_buttons = [dict(
        label="Totes les Espècies",
        method="update",
        args=[{"visible": [True] * len(species_list)},
              {"title": "Relació Activitat / Temperatura: Totes les Espècies"}]
    )]

    # Un botó per a cada espècie individual
    for i, sp in enumerate(species_list):
        visible_mask = [False] * len(species_list)
        visible_mask[i] = True
        species_buttons.append(dict(
            label=sp,
            method="update",
            args=[{"visible": visible_mask},
                  {"title": f"Relació Activitat / Temperatura: {sp}"}]
        ))

    # 4. Ajustos del disseny (Layout)
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=species_buttons,
                direction="down",
                showactive=True,
                x=0.01,
                y=1.12,
                xanchor="left",
                yanchor="top",
                bgcolor="#333", # Fons fosc per al menú
                font=dict(color="white")
            ),
        ],
        title="Relació Activitat / Temperatura: Totes les Espècies",
        xaxis_title="Temperatura Ambiental (°C)",
        yaxis_title="Número de Deteccions (Count)",
        template="plotly_dark",
        height=700,
        margin=dict(t=100) # Espai superior per al menú
    )

    fig.show()

# Per executar-la:
plot_observations_vs_temperature_interactive(df)